<sub>🐍 **Python for Django** · by **Rifat Islam Sourov** · Chapter 11</sub>

# 📝 Project 2: Build a Mini Blog (Django-style!)

This is the **most important project** in the course. We'll build a tiny blog engine in **pure Python** that works **the same way Django does**.

When you start learning Django, you'll think: *"Hey, I already built this!"* 😎

## 🧩 The pieces (Django has the same ones!)

| Our piece | What it does | Django name |
|-----------|--------------|-------------|
| `Author`, `Post` classes | describe our data | **Models** |
| `PostManager` | finds and filters posts | **`Post.objects`** (QuerySets) |
| `render_...` functions | turn data into HTML | **Templates** |
| `home()`, `post_detail()` | handle a page visit | **Views** |
| `urlpatterns` | match a URL to a view | **URLconf** (`urls.py`) |

Each step has an empty cell for you and a hidden solution. Let's go! 🚀

---
## Step 1: The `Author` model 👤
Create a class `Author` with `name` and `email`, and a `__str__` that returns the name.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
class Author:
    def __init__(self, name, email):
        self.name = name
        self.email = email

    def __str__(self):
        return self.name

rifat = Author("Rifat", "rifat@example.com")
sara = Author("Sara", "sara@example.com")
print(rifat, sara)
```

</details>

## Step 2: The `Post` model 📰
Create a class `Post` with:
- `title`, `content`, `author` (an `Author` object!)
- `slug`, made **automatically** from the title (`"Hello World"` → `"hello-world"`)
- `is_published = False` and `created_at = datetime.now()`
- a `publish()` method that sets `is_published` to `True`
- a `__str__` that returns the title

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
from datetime import datetime

class Post:
    def __init__(self, title, content, author):
        self.title = title
        self.content = content
        self.author = author
        self.slug = title.strip().lower().replace(" ", "-")
        self.is_published = False
        self.created_at = datetime.now()

    def publish(self):
        self.is_published = True

    def __str__(self):
        return self.title

post = Post("Hello World", "My very first post!", rifat)
print(post, "|", post.slug, "|", post.author, "|", post.is_published)
```

</details>

## Step 3: The `PostManager`, like `Post.objects` 🗂️
In Django you write `Post.objects.all()` and `Post.objects.filter(...)`. Let's build our own!

Create a class `PostManager` with:
- `self.posts = []`
- `create(title, content, author)`: makes a `Post`, adds it to the list and returns it
- `all()`: returns all posts
- `filter(**kwargs)`: returns posts where **every** given attribute matches, e.g. `filter(is_published=True)`
- `get(slug)`: returns the post with that slug, or raises a `PostNotFound` error

💡 Hint for `filter`: `getattr(post, "title")` is the same as `post.title`.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
class PostNotFound(Exception):
    pass

class PostManager:
    def __init__(self):
        self.posts = []

    def create(self, title, content, author):
        post = Post(title, content, author)
        self.posts.append(post)
        return post

    def all(self):
        return self.posts

    def filter(self, **kwargs):
        results = []
        for post in self.posts:
            if all(getattr(post, key) == value for key, value in kwargs.items()):
                results.append(post)
        return results

    def get(self, slug):
        for post in self.posts:
            if post.slug == slug:
                return post
        raise PostNotFound(f"No post with slug '{slug}'")

objects = PostManager()
objects.create("Hello World", "My very first post!", rifat).publish()
objects.create("Learning Python", "Python is so friendly.", sara).publish()
objects.create("Django Is Next", "Coming soon...", rifat)    # a draft, not published

print("All:", [str(p) for p in objects.all()])
print("Published:", [str(p) for p in objects.filter(is_published=True)])
print("By Rifat:", [str(p) for p in objects.filter(author=rifat)])
print("Get:", objects.get("learning-python"))
```

</details>

## Step 4: Templates, turning data into HTML 🎨
Websites send **HTML** to the browser. Write two functions that return HTML **strings**:

- `render_post_list(posts)`: an `<h1>` heading, then for every post an `<h2>` with a link like `<a href="/post/hello-world/">Hello World</a>` and a `<p>` with the author
- `render_post_detail(post)`: the title, author, date (`"24 Oct 2026"`) and content

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
def render_post_list(posts):
    html = "<h1>📝 My Mini Blog</h1>\n"
    for post in posts:
        html += f'<h2><a href="/post/{post.slug}/">{post.title}</a></h2>\n'
        html += f"<p>by {post.author}</p>\n"
    return html

def render_post_detail(post):
    return (
        f"<h1>{post.title}</h1>\n"
        f"<p>by {post.author} on {post.created_at.strftime('%d %b %Y')}</p>\n"
        f"<div>{post.content}</div>"
    )

print(render_post_list(objects.filter(is_published=True)))
print(render_post_detail(objects.get("hello-world")))
```

</details>

## Step 5: Views, the functions that handle a visit 👀
A **view** takes a `request` and returns a **response**. Our `request` is a simple dictionary like `{"path": "/", "user": None}`.

Write:
- `home(request)`: returns the list page with **published** posts only
- `post_detail(request, slug)`: returns the detail page, or `"404 😢 Post not found"` if the slug doesn't exist
- `create_post(request, title, content)`: only works if `request["user"]` is an `Author`; otherwise returns `"🔒 Please log in"`. **Use a decorator!** (Chapter 09)

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
from functools import wraps

def login_required(view):
    @wraps(view)
    def wrapper(request, *args, **kwargs):
        if not isinstance(request.get("user"), Author):
            return "🔒 Please log in"
        return view(request, *args, **kwargs)
    return wrapper

def home(request):
    return render_post_list(objects.filter(is_published=True))

def post_detail(request, slug):
    try:
        post = objects.get(slug)
    except PostNotFound:
        return "404 😢 Post not found"
    return render_post_detail(post)

@login_required
def create_post(request, title, content):
    post = objects.create(title, content, request["user"])
    post.publish()
    return f"✅ Created '{post.title}' at /post/{post.slug}/"

guest_request = {"path": "/", "user": None}
rifat_request = {"path": "/", "user": rifat}

print(home(guest_request))
print(post_detail(guest_request, "no-such-post"))
print(create_post(guest_request, "Hacked!", "..."))
print(create_post(rifat_request, "Views Are Functions", "Now I get it!"))
```

</details>

## Step 6: URLs, connecting addresses to views 🗺️
The last piece! When someone visits `/post/hello-world/`, Django looks through a list of **URL patterns** to find the right view.

Write:
- `urlpatterns`: a list of `(prefix, view)` tuples: `("/post/", post_detail)` and `("/", home)`
- `handle(path, user=None)`: builds a `request`, finds the matching view and returns its response. For `/post/<slug>/`, pass the slug to the view.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
urlpatterns = [
    ("/post/", post_detail),
    ("/", home),
]

def handle(path, user=None):
    request = {"path": path, "user": user}
    for prefix, view in urlpatterns:
        if prefix == "/" and path == "/":
            return view(request)
        if prefix != "/" and path.startswith(prefix):
            slug = path[len(prefix):].strip("/")
            return view(request, slug)
    return "404 😢 Page not found"

for path in ["/", "/post/views-are-functions/", "/post/django-is-next/", "/contact/"]:
    print(f"🌐 GET {path}")
    print(handle(path))
    print("-" * 40)
```

</details>

Wait... `/post/django-is-next/` showed the post even though it's a **draft**! 🐛

## Step 7: Fix the bug 🐞
Change `post_detail` so that **unpublished** posts also return the 404 message. Then run `handle("/post/django-is-next/")` again.

In [ ]:
# ✍️ Your code here

<details>
<summary>👀 <b>Show solution</b> (try it yourself first!)</summary>

```python
def post_detail(request, slug):
    try:
        post = objects.get(slug)
    except PostNotFound:
        return "404 😢 Post not found"
    if not post.is_published:
        return "404 😢 Post not found"
    return render_post_detail(post)

urlpatterns = [
    ("/post/", post_detail),
    ("/", home),
]
print(handle("/post/django-is-next/"))
print(handle("/post/hello-world/"))
```

</details>

---
## 🎉 YOU BUILT A WEB FRAMEWORK!
Seriously. Here's how your code compares to real Django:

| You wrote | Django version |
|-----------|----------------|
| `class Post:` | `class Post(models.Model):` |
| `objects.filter(is_published=True)` | `Post.objects.filter(is_published=True)` |
| `objects.get("hello-world")` | `Post.objects.get(slug="hello-world")` |
| `raise PostNotFound` | `Post.DoesNotExist` / `Http404` |
| `render_post_list(posts)` | `render(request, "post_list.html", {"posts": posts})` |
| `def home(request):` | `def home(request):` ← **exactly the same!** |
| `@login_required` | `@login_required` ← **exactly the same!** |
| `urlpatterns = [...]` | `urlpatterns = [path("", home), ...]` |

The difference? Django saves posts in a real **database**, uses real HTML **template files**, and runs a real **web server**. The *ideas* are exactly what you just built. 🧠

**➡️ Next: the Bridge to Django!**

---
### ⏭️ Next up: [🌉 Bridge to Django: You're Ready!](../12-Bridge-to-Django/01-From-Python-to-Django.ipynb)

<sub>Made with ❤️ by Rifat Islam Sourov · [Back to the course home](../README.md)</sub>